# 사전 등록 검증: 지자체 인구정책 뉴스레터의 주력 주제 고르기

- **사전 등록:** [`preregistration_niche_topic.md`](preregistration_niche_topic.md) · [`niche_candidates.py`](niche_candidates.py) — 커밋 `0f41961`, 후보 B~F 값을 받기 **전**.
- 모든 후보: 같은 범용 파이프라인, 통일 규칙 6차. 통과 = 전체·모든 규모 구간 잘못된 경보 ≤ 10%.
- 점수 = 감지율 × 커버 시군구 수(2025). 1위가 현재 주제(청년 순이동, 전입 2천+) 점수 × 1.1 이상이면 주력 주제 교체.
- 후보 A는 이미 본 자료(청년 순이동)라 해석할 때 표시합니다.

In [1]:
import os, urllib.request, warnings; warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/quanter02/chelsea-analytics/claude/great-dirac-3wwgzy/experiments/"
for f in ["kosis_monitor.py", "niche_candidates.py", "track_record.py"]:
    if not os.path.exists(f): urllib.request.urlretrieve(RAW + f, f)
import pandas as pd, kosis_monitor as K, niche_candidates as N, track_record as TR
outs = {name: K.run(tp, rule="v6", verbose=False) for name, tp in N.CANDIDATES.items()}
base = K.run(K.TOPICS[N.BASELINE], rule="v6", verbose=False)

## 1. 공식 채점 (채점 기간 2018~2025)

In [2]:
def gun(out, year=2025):
    tp = out["topic"]
    return sum(1 for (u, y), d in out["units"].items() if y == year and d["size"] >= tp.min_size and str(d["name"]).endswith("군"))
rows = {name: {**N.score(o), "군_커버": gun(o), "α": o["calibration"].alpha, "m": round(o["calibration"].m, 3)} for name, o in outs.items()}
S = pd.DataFrame(rows).T
b = {**N.score(base), "군_커버": gun(base)}
show = S.copy()
for c in ["감지율", "잘못된경보", "구간최대_잘못된경보", "상반기내"]: show[c] = show[c].map(lambda v: f"{float(v):.1%}")
show["점수"] = show["점수"].map(lambda v: f"{float(v):.1f}")
print(f"기준(현재 주제 {N.BASELINE}): 감지율 {b['감지율']:.1%}, 커버 {b['커버_시군구']}곳(군 {b['군_커버']}), 점수 {b['점수']:.1f}")
show

기준(현재 주제 청년 순이동): 감지율 84.8%, 커버 157곳(군 13), 점수 0.0


,통과,감지율,잘못된경보,구간최대_잘못된경보,상반기내,커버_시군구,점수,군_커버,α,m
A 청년 순이동(전입 300+),False,68.9%,7.5%,11.5%,37.5%,228,0.0,81,0.99,0.0
B 영유아(0~9세) 순이동,True,77.6%,5.5%,9.7%,41.5%,163,126.5,18,0.99,0.067
C 청년 여성 순이동,False,66.4%,6.6%,10.1%,39.3%,218,0.0,71,0.99,0.049
D 고령(65+) 순이동,False,81.2%,5.3%,21.7%,42.0%,216,0.0,69,0.99,0.071
E 전체 순이동,True,71.5%,5.1%,6.6%,41.0%,228,163.0,81,0.99,0.0
F 주민등록인구,True,39.6%,0.4%,0.5%,18.0%,275,108.9,82,0.99,0.0


In [3]:
choice, why = N.rank(S.assign(통과=S.통과.astype(bool), 점수=S.점수.astype(float)), float(b["점수"]))
print("결정:", why)

결정: E 전체 순이동 채택 (점수 163.0 ≥ 기준 0.0 × 1.1)


## 2. 규모 구간별 (작은 지역에서도 기준을 지키는지)

In [4]:
for name, o in outs.items():
    print(name); display(o["groups"])

A 청년 순이동(전입 300+)


,단위연도,이탈해,감지율,정상해,잘못된경보율
규모,,,,,
300~2천,511.0,207.0,0.391,146.0,0.007
2천~1만,504.0,170.0,0.776,180.0,0.033
1만 이상,809.0,154.0,0.994,425.0,0.115


B 영유아(0~9세) 순이동


,단위연도,이탈해,감지율,정상해,잘못된경보율
규모,,,,,
300~2천,660.0,215.0,0.623,247.0,0.020
2천~1만,696.0,194.0,0.923,335.0,0.078
1만 이상,72.0,20.0,1.000,31.0,0.097


C 청년 여성 순이동


...

,단위연도,이탈해,감지율,정상해,잘못된경보율
규모,,,,,
300~2천,911.0,276.0,0.721,337.0,0.036
2천~1만,790.0,158.0,0.968,401.0,0.057
1만 이상,32.0,2.0,1.000,23.0,0.217


E 전체 순이동


,단위연도,이탈해,감지율,정상해,잘못된경보율
규모,,,,,
300~2천,32.0,13.0,0.538,12.0,0.000
2천~1만,633.0,188.0,0.495,237.0,0.021
1만 이상,1161.0,297.0,0.862,531.0,0.066


F 주민등록인구


,단위연도,이탈해,감지율,정상해,잘못된경보율
규모,,,,,
3만 미만,176.0,47.0,0.000,63.0,0.000
3만~10만,594.0,208.0,0.096,146.0,0.000
10만 이상,1352.0,333.0,0.640,575.0,0.005


## 3. 적중 기록 미리 재기 (결정에 쓰지 않음)
채점 기간에 매달 발행했다면 3·6·9월호 경보가 연말 확정치로 몇 % 맞았나. 정의는 `track_record.py`.

In [5]:
P = []
for name, o in {**outs, f"(기준) {N.BASELINE}": base}.items():
    p = TR.precision(o); p.insert(0, "주제", name); P.append(p)
pd.concat(P, ignore_index=True)

,주제,발행월,연평균_경보,같은방향,진짜이탈,평범한해
0,A 청년 순이동(전입 300+),3월,16.0,0.969,0.703,0.078
1,A 청년 순이동(전입 300+),6월,39.6,0.981,0.628,0.101
2,A 청년 순이동(전입 300+),9월,59.1,0.992,0.617,0.101
3,B 영유아(0~9세) 순이동,3월,16.2,0.938,0.731,0.100
4,B 영유아(0~9세) 순이동,6월,30.6,0.963,0.722,0.094
5,B 영유아(0~9세) 순이동,9월,44.5,0.989,0.739,0.059
6,C 청년 여성 순이동,3월,18.1,0.959,0.690,0.110
7,C 청년 여성 순이동,6월,41.4,0.979,0.653,0.097
8,C 청년 여성 순이동,9월,57.9,0.994,0.657,0.069
9,D 고령(65+) 순이동,3월,12.4,0.909,0.646,0.182


## 4. 탐색: 주제를 넘나드는 신호 (사전 등록 아님, 가설 만들기용)
2026년 실시간 경보가 여러 이동 주제에 겹치는 지역. 같은 자료(전입·전출)의 부분집합끼리라 겹치는 것이 당연한 부분도 있으므로 해석에 주의합니다.

In [6]:
live = []
for name, o in {**outs}.items():
    L = o["live"]
    if len(L): live.append(L[L.상태 == "유지"].assign(주제=name)[["주제", "지역", "방향", "이탈률"]])
X = pd.concat(live, ignore_index=True)
piv = X.pivot_table(index="지역", columns="주제", values="방향", aggfunc="first")
piv["경보_주제수"] = piv.notna().sum(1)
piv.sort_values("경보_주제수", ascending=False).head(20)

주제,A 청년 순이동(전입 300+),B 영유아(0~9세) 순이동,C 청년 여성 순이동,D 고령(65+) 순이동,E 전체 순이동,F 주민등록인구,경보_주제수
지역,,,,,,,
대전 유성구,증가,증가,증가,증가,증가,증가,6
대구 남구,증가,증가,증가,증가,증가,증가,6
서울 송파구,증가,증가,증가,증가,증가,증가,6
부산 남구,증가,증가,증가,증가,증가,NaN,5
경기 용인시,감소,감소,감소,감소,감소,NaN,5
서울 강동구,감소,감소,감소,감소,감소,NaN,5
인천 중구,감소,감소,감소,감소,감소,NaN,5
부산 부산진구,감소,감소,감소,NaN,감소,NaN,4
서울 성북구,감소,NaN,감소,감소,감소,NaN,4
